# Data Generation Pipeline in a Nutshell

The thing we want to attain is splitting the original data set (that which
we call the _golden master_) into two or more subsets _**by feature**_.

As a first step, we're going to load the original data set into memory by
using the standard Python tooling. This needs to be extended to support a never-
ending stream of CSV records.

In [1]:
FILE_NAME = "ACM.csv"

In [2]:
from csv import reader

with open(FILE_NAME, "r") as csv_file:
    lines = csv_file.readlines()

csv_reader = reader(lines)

Next, we define a couple of utility functions that help us manipulate the
in-memory data as if it were iterable.

In [3]:
from itertools import islice

def skip(seq, count):
    return islice(seq, count, None)

def take(seq, count):
    return islice(seq, count)

The features are determined by the first (title) row of the csv input. Let's
create the 'feature' -> 'index' mapping so we know later on what we're reading.

In [4]:
title_row = next(take(csv_reader, 1))
feature_map = {feature: idx for idx, feature in enumerate(title_row)}
print(feature_map)

{'id': 0, 'title': 1, 'authors': 2, 'venue': 3, 'year': 4}


Next, let's set up a couple of parameters.

In [5]:
# features that are going to be found across all datasets
fixed_features = ["id"]
# number of datasets that we want to generate
output_dataset_count = 2
# min and max number of features of the resulting datasets
min_output_features = 1
max_output_features = 3

Next, we set up the columns we need to extract

In [6]:
from collections import namedtuple
from functools import reduce
from random import randint, sample

all_feature_names = set(feature_map.keys())
eligible_features = list(all_feature_names - set(fixed_features))
feature_lists = []
for i in range(output_dataset_count):
    n = len(eligible_features)
    extra_feature_count = randint(min_output_features, max_output_features)
    retry = 3
    while n < extra_feature_count and retry > 0:
        extra_feature_count = randint(min_output_features, max_output_features)
        retry -= 1
    if n < extra_feature_count:
        raise ValueError("Bad config - can't split datasets")

    extra_features = list(sample(eligible_features, extra_feature_count))
    feature_lists.append(extra_features + fixed_features)
    for feature in extra_features:
        eligible_features.remove(feature)


Now that that's over with, we can construct our generation output metadata.
This is where we store stuff that we'll need when we compare what the entity
resolution algorithm produced with our initial dataset (golden master).

For example, if we've somehow ignored some features in the generation phase
because they weren't picked up by our feature sampler, then those features
should be ignored when comparing results.

In [7]:
OutputMetadata = namedtuple("OutputMetadata", [
    "feature_lists",
    "all_features",
    "generated_features",
    "ignored_features"
])

generated_features = reduce(
    lambda acc, x: acc.union(set(x)),
    feature_lists,
    set()
)
generator_metadata = OutputMetadata(
    feature_lists=feature_lists,
    all_features=list(all_feature_names),
    generated_features=list(generated_features),
    ignored_features=list(all_feature_names - generated_features)
)
print(generator_metadata)

OutputMetadata(feature_lists=[['title', 'venue', 'authors', 'id'], ['year', 'id']], all_features=['id', 'year', 'venue', 'authors', 'title'], generated_features=['venue', 'authors', 'year', 'id', 'title'], ignored_features=[])


Next up, we're going to actually generate the new data sets.

In [8]:
output = []

def generate_records(record, feature_lists):
    yield [
        tuple(record[feature_map[x]] for x in features)
        for features in feature_lists
    ]

def add_to_dataset(data_sets, idx, value):
    n = len(data_sets)
    if idx < 0:
        idx = 0
        data_sets.insert(0, [])
    if idx >= n:
        idx = n
        data_sets.append([])
    data_sets[idx].append(value)

data_sets = []    
for record in csv_reader:
    for generated in generate_records(record, generator_metadata.feature_lists):
        for idx, generated_record in enumerate(generated):
            add_to_dataset(data_sets, idx, generated_record)
for idx, ds in enumerate(data_sets):
    print("dataset", idx, generator_metadata.feature_lists[idx])
    print(list(take(ds, 2)))


dataset 0 ['title', 'venue', 'authors', 'id']
[('The WASA2 object-oriented workflow management system', 'International Conference on Management of Data', 'Gottfried Vossen, Mathias Weske', '304586'), ('A user-centered interface for querying distributed multimedia databases', 'International Conference on Management of Data', 'Isabel F. Cruz, Kimberly M. James', '304587')]
dataset 1 ['year', 'id']
[('1999', '304586'), ('1999', '304587')]


Now we have our resulting datasets. All we need to do is stored them in a format
that the entity resolution algorithm can consume.